# **Notebook 3: Modelling**

In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sb
import warnings as warnings

In [25]:
dataset=pd.read_csv('df_processed.csv')

In [26]:
dataset['rh']

0          94.0
1          86.7
2          74.4
3          97.0
4          93.2
           ... 
3159371    65.0
3159372    59.0
3159373    61.0
3159374    59.0
3159375    39.0
Name: rh, Length: 3159376, dtype: float64

In [27]:
X, y=dataset[['temp', 'rh', 'ws', 'precip', 'rndays']], dataset['risk_band']

In [28]:
X.head()

,temp,rh,ws,precip,rndays
0,9.5,94.0,8.6,20.52,0
1,1.0,86.7,11.1,1.71,0
2,-2.0,74.4,0.0,0.00,1
3,13.2,97.0,3.7,16.80,0
4,2.0,93.2,13.0,3.80,0


## **Train test split**

In [29]:
# How does stratify help? Ensures the training and testing sets have the same proportion of classes as the orignal dataset. Very important when dealing with inbalanced datasets. 
# Like our processed dataset that has risk_band=no_risk inflated observations
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.3, random_state=34, stratify=y)
X_train

,temp,rh,ws,precip,rndays
327162,20.2,65.0,13.4,14.60,0
310872,6.4,76.0,2.1,0.10,0
1899565,25.0,29.6,9.2,0.00,5
2348670,-28.0,75.5,0.0,0.01,1
1614328,25.0,45.0,4.3,0.00,9
...,...,...,...,...,...
2734004,-1.7,72.2,9.3,0.00,3
1762668,21.8,48.0,18.3,0.00,1
2625672,-0.1,50.0,9.3,0.00,5
162459,15.0,96.0,15.0,26.40,0


## **Feature Scaling**

### Since LDA/QDA/KNN and classification techniques of the like are distance/variance sensitive therefore feature scaling is important. 
### unscaled features can distort the geometry the model relies on. 

In [30]:
scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

### **Fitting Logistic Regression**

In [31]:
model=LogisticRegression(solver='lbfgs', max_iter=200)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,200
,multi_class,'deprecated'


In [32]:
y_pred=model.predict(X_test_scaled)

#### Fitting a multinominal logistic regression model with and without no_risk. This was to check model performance even when an easily predictable class was not present. 

In [34]:
print(classification_report(y_test, y_pred, target_names=['extreme', 'high', 'moderate', 'no_risk', 'very_high', ]))

              precision    recall  f1-score   support

     extreme       0.78      0.65      0.71     27420
        high       0.53      0.36      0.43    137450
    moderate       0.73      0.80      0.76    352421
     no_risk       0.81      0.82      0.81    321469
   very_high       0.59      0.65      0.62    109053

    accuracy                           0.72    947813
   macro avg       0.69      0.65      0.67    947813
weighted avg       0.71      0.72      0.71    947813



#### As expected, no_risk is the best performing class with a precision of 0.81. 

In [35]:
print(classification_report(y_test, y_pred, labels=['extreme','high','moderate','very_high'], target_names=['extreme','high','moderate','very_high']))

              precision    recall  f1-score   support

     extreme       0.78      0.65      0.71     27420
        high       0.53      0.36      0.43    137450
    moderate       0.73      0.80      0.76    352421
   very_high       0.59      0.65      0.62    109053

   micro avg       0.67      0.67      0.67    626344
   macro avg       0.66      0.61      0.63    626344
weighted avg       0.66      0.67      0.66    626344



#### Whats more significant to notice is the f1 score dropping from 0.67 to 0.63. This means, no_risk was responsible of inflating the overall macro-f1 average. Excluding it reveals the models skill at distinguishing risk severities which is lower and more relecant to the projects actual goal. 